In [1]:
import pandas as pd
import numpy as np
from dateutil.parser import parse

#import and read data
samples = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/AmostrasAngolaTerrario.xlsx")
analyses = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Horizontes Analises.xlsx")
morphology = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Horizontes_Morfologia.xlsx")
profile_loc = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Perfis_local.xlsx")
soil_profile = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Perfis_solo.xlsx")
elemental_analyses = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Data XRF Angola_inicial.xlsx")
#soil_type = pd.read_excel("/Users/inesschwartz/Desktop/Thesis/tables_soil_database/Perfis_solo.xlsx")

# Samples table

In [2]:
samples = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/AmostrasAngolaTerrario.xlsx")

# Rename columns in the DataFrame
samples.rename(columns={
    'Registo': 'sample_id',
    'Nº Campo': 'site_info_id',
    'Ano': 'year',
    'Perfil': 'profile',
    'Campanha': 'campaign',
    'Colónia_Pais': 'country',
    'Distrito': 'district',
    'AmostraCrivada': 'sample_sifted',
    'AmostraNaoCrivada': 'sample_not_sifted',
    'Prateleira': 'shelf',
    'Sala': 'room'
}, inplace=True)

# Replace nulls with the string 'null'
samples_filled = samples.fillna('null')

# analyse duplicated sample_id (PK)
duplicated_values = samples['sample_id'][samples['sample_id'].duplicated()].unique()
print(duplicated_values)

# Change PK (sample_id) to a consistent datatype: Converted to string and truncate to 20 characters (similar to VARCHAR(20))
samples_filled['sample_id'] = samples_filled['sample_id'].astype(str).str.strip().str[:20]

# Drop unnecessary columns
samples_clean = samples_filled.drop(columns=[
    'campaign', 'country', 'Província', 'sample_not_sifted', 'sample_sifted', 'Obs'
])

samples_clean = samples_filled

## add empty FK columns to populate later
samples_clean['lab_info_id'] = pd.NA
samples_clean['horizon_id'] = pd.NA
samples_clean['profile_record_id'] = pd.NA


# Rename for consistent schema (only if needed)
samples_clean = samples_clean.rename(columns={
    'profile': 'profile',
    'shelf': 'shelf',
    'room': 'room',
    'year': 'year'
})

samples_clean['shelf'] = samples_clean['shelf'].astype("string")

samples_clean['profile'] = samples_clean['profile'].astype(str).str.strip().str[:20]



# Reorder columns to match SAMPLES schema
samples_check = samples_clean[[
    'sample_id',
    'site_info_id',
    'profile_record_id',
    'profile',
    'horizon_id',
    'shelf',
    'room',
    'year'
]]

## Preview results:
samples_check.head()

[]


,sample_id,site_info_id,profile_record_id,profile,horizon_id,shelf,room,year
0,630,172,<NA>,139,<NA>,1,22,1946.0
1,631,173,<NA>,139,<NA>,1,22,1946.0
2,632,174,<NA>,139,<NA>,1,22,1946.0
3,633,175,<NA>,139,<NA>,1,22,1946.0
4,687,1034,<NA>,208,<NA>,1,22,1946.0


# Analyses table

In [3]:
analyses.rename(columns={
    'Amostra': 'sample_id',
    'Morfo_id': 'horizon_id',
    'Analise-id': 'analysis_id',
    'PERFIL': 'profile', #change to reference PK profile_record_id
    'LS': 'upper_limit',
    'LI': 'lower_limit',
    'EG': 'EG',
    'AG': 'thick_clay',
    'AF': 'fine_clay',
    'Argila': 'clay',
    'L': 'silt',
    'Gesso': 'gypsum',
    'Fe livre': 'free_iron',
    'CO':'organic_carbon',
    'N total': 'total_N',
    'MO': 'OM',
    'Soma de bases': 'exchangable_bases_sum',
    'CTC': 'CEC',
    'Cloretos':'chlorides',
    'Sulfatos':'sulfates',
    'Condutividade':'conductivity',
    'Sódio solúvel':'soluble_sodium',
    'P205 total': 'P205',
    'pH (H2O)':'pH_H2O',
    'pH(KCL)':'pH_KCL',
    'MO': 'organic_material'
}, inplace=True)

# Replace nulls with the string 'null'
analyses_filled = analyses.fillna('null')

# Filter out and print duplicated samples (including the first occurrence)
duplicated_values = analyses['sample_id'][analyses['sample_id'].duplicated()].unique()
print(duplicated_values)

# Drop columns not needed
analyses_drop = analyses.drop([
    'EqMol (SiO2)', 'EqMol(Al2O3)', 'EqMol(Fe2O3)', 'SiO2/Al2O3', 
    'SiO2/Fe2O3', 'SiO2/R2O3', 'Fe2O3/Al2O3', 'FE2O3_TARG', 
    'FE2O3_LARG', 'CEC_ARG', 'GR', 'ID', 'COD_PROV'
], axis=1)

# Add a new Primary Key ID column starting from 1
analyses_drop.insert(0, 'lab_sample_id', range(1, len(analyses_drop) + 1))

# Final cleaned dataframe
analyses2 = analyses_drop

# Define the desired final column order
final_columns = [
    'lab_sample_id',
    'sample_id',
    'minerology_id',
    'EG',
    'thick_clay',
    'fine_clay',
    'silt',
    'clay',
    'Eq_Hum',
    'atm_1/3',
    'atm_15',
    'CACO3',
    'gypsum',
    'free_iron',
    'organic_carbon',
    'total_N',
    'P205',
    'organic_material',
    'pH_H2O',
    'pH_KCL',
    'Ca++',
    'Mg++',
    'Na+',
    'K+',
    'exchangable_bases_sum',
    'CEC',
    'V',
    'conductivity',
    'soluble_sodium',
    'Min_<0,002',
    'Min_0,05-0,02',
    'Min_0,2-0,05',
    'Min_2-0,2',
]


# Add missing columns with NaN values
for col in final_columns:
    if col not in analyses2.columns:
        analyses2[col] = pd.NA  # or np.nan if you prefer

# Reorder columns
analyses_ready = analyses2[final_columns]

# Preview first 10 rows
analyses_ready.head(10)


[10337.    nan 11608.  3497.  3498.  2923.  2924.  2925.  2926.  2927.
  2928.  2929.  2930. 16355. 16356.   419.   420. 15682. 13817. 13965.
  3418.  9230.  8639.  3529.]


,lab_sample_id,sample_id,minerology_id,EG,thick_clay,fine_clay,silt,clay,Eq_Hum,atm_1/3,...,K+,exchangable_bases_sum,CEC,V,conductivity,soluble_sodium,"Min_<0,002","Min_0,05-0,02","Min_0,2-0,05","Min_2-0,2"
0,1,10999.0,<NA>,NaN,61.700001,32.8,0.2,5.3,4.600000,NaN,...,0.03,NaN,1.83,23.000000,NaN,0.0,NaN,NaN,NaN,NaN
1,2,11000.0,<NA>,NaN,52.799999,35.1,0.7,11.4,6.400000,NaN,...,0.03,NaN,1.98,10.600000,NaN,0.0,NaN,NaN,NaN,NaN
2,3,11001.0,<NA>,NaN,42.500000,46.2,0.2,11.1,6.300000,NaN,...,0.01,NaN,1.62,5.600000,NaN,0.0,NaN,NaN,NaN,NaN
3,4,11002.0,<NA>,NaN,42.599998,41.8,0.2,15.4,5.200000,NaN,...,0.01,NaN,0.91,8.800000,NaN,0.0,NaN,NaN,NaN,NaN
4,5,11003.0,<NA>,NaN,36.799999,47.5,1.2,14.5,7.300000,NaN,...,0.01,NaN,1.04,7.700000,NaN,0.0,NaN,NaN,NaN,NaN
5,6,11004.0,<NA>,NaN,45.099998,41.8,1.4,11.7,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
6,7,11011.0,<NA>,1.0,15.300000,48.5,17.6,18.5,19.700001,NaN,...,0.18,NaN,4.49,32.500000,NaN,0.0,NaN,NaN,NaN,NaN
7,8,11012.0,<NA>,3.0,18.600000,42.3,17.8,21.3,19.600000,NaN,...,0.07,NaN,2.96,20.600000,NaN,0.0,NaN,NaN,NaN,NaN
8,9,11013.0,<NA>,7.0,12.000000,36.8,22.1,29.1,21.700001,NaN,...,0.04,NaN,3.02,16.900000,NaN,0.0,NaN,NaN,NaN,NaN
9,10,11014.0,<NA>,21.0,11.000000,34.8,21.6,32.6,21.600000,NaN,...,0.03,NaN,3.11,19.299999,NaN,0.0,NaN,NaN,NaN,NaN


## joining elemental analyses to analyses


In [4]:
# Rename 'Lab. Code' to 'sample_id'
elemental_analyses.rename(columns={'Lab. Code': 'sample_id'}, inplace=True)

# Temporarily convert to string to do the replacements
sample_id_cleaned = elemental_analyses['sample_id'].astype(str)
sample_id_cleaned = sample_id_cleaned.str.replace('C-', '', regex=False)
sample_id_cleaned = sample_id_cleaned.str.replace(' MNL', '', regex=False)

# Then convert to numeric (int or float, depending on your data)
elemental_analyses['sample_id'] = pd.to_numeric(sample_id_cleaned, errors='coerce')

#rename to field_sample_code
elemental_analyses.rename(columns={
    'Code': 'field_sample_code'
}, inplace=True)

In [5]:
print (analyses_ready['sample_id'].dtype)
print (elemental_analyses['sample_id'].dtype)


float64
int64


In [6]:
elemental_analyses['sample_id'] = elemental_analyses['sample_id'].astype(float)


In [7]:
# Check which sample_ids in elemental_analyses are NOT in analyses_ready
missing_ids = elemental_analyses[~elemental_analyses['sample_id'].isin(analyses_ready['sample_id'])]

# Display them
print("Sample IDs in elemental_analyses not found in analyses_ready:")
print(missing_ids['sample_id'].unique())


Sample IDs in elemental_analyses not found in analyses_ready:
[16105. 16217. 16033. 16231. 16208. 16274. 16225. 15731. 16078. 15786.
 15693. 15678. 15959. 16372. 16418. 16459. 16477. 15498. 15437. 12749.
 13109. 14337. 14319. 15484. 15508. 15614. 15580. 15582. 15606. 15463.
 14496. 11343. 17892. 16976. 17269. 18248. 16946. 17686. 17728. 17415.
 17642. 18212. 17016. 17282. 18429. 17340. 18824. 18448.  8721.  7313.
  5502.  5392.  7317.  8626.  7193.  7254.  7293.  8240. 15397. 14403.
 14983. 16126. 15059. 15322. 15268. 14808. 15106.]


LEFT JOIN ELEMENTAL ANALYSE TO ANALYSE TABLE

In [8]:
merged1= analyses_ready.merge(elemental_analyses, on='sample_id', how='left')
merged1.head()


,lab_sample_id,sample_id,minerology_id,EG,thick_clay,fine_clay,silt,clay,Eq_Hum,atm_1/3,...,Ta,W,Pt,Au,Hg,Tl,Pb,Bi,Th,U
0,1,10999.0,<NA>,NaN,61.700001,32.8,0.2,5.3,4.6,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,11000.0,<NA>,NaN,52.799999,35.1,0.7,11.4,6.4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,11001.0,<NA>,NaN,42.500000,46.2,0.2,11.1,6.3,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,11002.0,<NA>,NaN,42.599998,41.8,0.2,15.4,5.2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,11003.0,<NA>,NaN,36.799999,47.5,1.2,14.5,7.3,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(analyses_ready[analyses_ready['sample_id'] == 6940])
print(elemental_analyses[elemental_analyses['sample_id'] == 6940])


      lab_sample_id  sample_id minerology_id  EG  thick_clay  fine_clay  silt  \
1361           1362     6940.0          <NA> NaN        31.1       43.8   2.7   

      clay  Eq_Hum  atm_1/3  ...  K+  exchangable_bases_sum  CEC   V  \
1361  22.7    18.9      NaN  ... NaN                    NaN  NaN NaN   

      conductivity  soluble_sodium  Min_<0,002  Min_0,05-0,02  Min_0,2-0,05  \
1361           NaN             0.0         NaN            NaN           NaN   

      Min_2-0,2  
1361        NaN  

[1 rows x 33 columns]
    sample_id Soil Profile field_sample_code Code.1      Depht Sampling Date  \
80     6940.0      P134/59          A-320/59  S-492  1.30-1.70    1954-08-06   

             Mg             Al             Si      P  ...    Ta   W  Pt  Au  \
80  1457.666667  106266.333333  308859.666667  468.0  ...  26.0 NaN NaN NaN   

           Hg   Tl         Pb  Bi         Th   U  
80  10.666667  4.0  25.333333 NaN  12.333333 NaN  

[1 rows x 44 columns]


In [10]:
# Define final column list
final_columns = [
    'lab_sample_id', 'sample_id', 'EG', 'thick_clay', 'fine_clay', 'silt', 'clay', 'Eq_Hum', 'atm_1/3', 'atm_15',
    'CACO3', 'gypsum', 'free_iron', 'organic_carbon', 'total_N', 'P205', 'organic_material', 'pH_H2O', 'pH_KCL',
    'Ca++', 'Mg++', 'Na+', 'K+', 'exchangable_bases_sum', 'CEC', 'V', 'conductivity', 'soluble_sodium', 'Min_<0,002',
    'Min_0,05-0,02', 'Min_0,2-0,05', 'Min_2-0,2', 
    'field_sample_code', 'Depth', 'Al', 'Si', 'P', 'S', 'Cl', 'Ti', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'As', 'Se', 'Rb', 'Sr', 'Zr', 'Nb', 'Mo', 'Cd', 'Sn', 'Sb', 'Ba', 'Ta', 'W', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi',
    'Th', 'U',
]

# Ensure all columns exist in the DataFrame
for col in final_columns:
    if col not in merged1.columns:
        merged1[col] = pd.NA  # or np.nan if preferred

# Reorder DataFrame columns
merged1 = merged1[final_columns]

# Check and print the datatypes of each column
print("Data types of each column in merged1:\n")
print(merged1.dtypes)

Data types of each column in merged1:

lab_sample_id      int64
sample_id        float64
EG               float64
thick_clay       float64
fine_clay        float64
                  ...   
Tl               float64
Pb               float64
Bi               float64
Th               float64
U                float64
Length: 68, dtype: object


### Check/Change datatypes

In [11]:
# Change PK (sample_id) to a consistent datatype: Converted to string and truncate to 20 characters (similar to VARCHAR(20))
merged1['sample_id'] = merged1['sample_id'].astype(str).str.strip().str[:20]
merged1['lab_sample_id'] = merged1['lab_sample_id'].astype(str).str.strip().str[:20]

print(merged1.dtypes)

lab_sample_id     object
sample_id         object
EG               float64
thick_clay       float64
fine_clay        float64
                  ...   
Tl               float64
Pb               float64
Bi               float64
Th               float64
U                float64
Length: 68, dtype: object


### Check and/or add FK

### Save to CSV

In [12]:
merged1.head()

,lab_sample_id,sample_id,EG,thick_clay,fine_clay,silt,clay,Eq_Hum,atm_1/3,atm_15,...,Ta,W,Pt,Au,Hg,Tl,Pb,Bi,Th,U
0,1,10999.0,NaN,61.700001,32.8,0.2,5.3,4.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,11000.0,NaN,52.799999,35.1,0.7,11.4,6.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,11001.0,NaN,42.500000,46.2,0.2,11.1,6.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,11002.0,NaN,42.599998,41.8,0.2,15.4,5.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,11003.0,NaN,36.799999,47.5,1.2,14.5,7.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
#merged1.to_csv("/Users/inesschwartz/GreenDataScience/Thesis/tables_clean/analyses_clean.csv", index=False)

# Soil Profile Table

In [14]:
## rename columns
soil_profile = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Perfis_solo.xlsx")

#rename columns
soil_profile.rename(columns={
    'Perfil': 'profile',
    'Agrupamento': 'grouping',
    'Pro': 'province',
    'País': 'country',
    'Local': 'location',
    'DATA': 'date',
    'CEP_NOME': 'CEP_NAME',
}, inplace=True)

# Drop unnecessary columns
soil_profile_cleaning1 = soil_profile.drop(columns=[
    'grouping', 'REF', 'province', 'country', 'LOCAL', 'DESCRITOR1', 'DESCRITOR2', 'DESCRITOR3', 'CEP_GR'
])

soil_profile_cleaning2 = soil_profile.drop(columns=[
    'grouping', 'REF', 'province', 'country', 'LOCAL', 'DESCRITOR1', 'DESCRITOR2', 'DESCRITOR3', 'date', 'CEP_GR'
])
# Add missing columns
soil_profile_cleaning2['site_info_id'] = pd.NA
soil_profile_cleaning2['sample_id'] = pd.NA 
soil_profile_cleaning2['soil_type_id'] = pd.NA 
soil_profile_cleaning2['site_info_id'] = soil_profile_cleaning2['site_info_id'].astype(str)

# Add a new Primary Key ID column starting from 1
soil_profile_cleaning2.insert(0, 'profile_record_id', range(1, len(soil_profile_cleaning2) + 1))

# Reorder columns to match SAMPLES schema
profile_record_clean = soil_profile_cleaning2[[
    'profile_record_id',
    'profile',
    'site_info_id',
    'sample_id',
    'soil_type_id'
]]

profile_record_clean['profile'] = profile_record_clean['profile'].astype(str).str.strip().str[:20]


# Preview the result
profile_record_clean.head()
#profile_clean.to_csv("/Users/inesschwartz/Desktop/Thesis/tables_clean/profile_record_clean.csv", index=False)


/var/folders/tp/79mdnyy56_xc3g1jvp9wf4_80000gn/T/ipykernel_3633/998777555.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  profile_record_clean['profile'] = profile_record_clean['profile'].astype(str).str.strip().str[:20]


,profile_record_id,profile,site_info_id,sample_id,soil_type_id
0,1,1/51,<NA>,<NA>,<NA>
1,2,1/57,<NA>,<NA>,<NA>
2,3,1/59,<NA>,<NA>,<NA>
3,4,1/63,<NA>,<NA>,<NA>
4,5,10/54,<NA>,<NA>,<NA>


# Morphology Horizon Table

In [15]:
#rename columns
morphology.rename(columns={
    'Morfo_id':'horizon_id',
    'Amostra': 'sample_id',
    'Perfil': 'profile',
    'CM':'horizon_layer',
    'Limite Superior': 'upper_depth',
    'Limite inferior': 'lower_depth',
    'Grau de humidade': 'moisture_degree',
    'Quantidade de raízes': 'root_quantity',
    'Diâmetro de raízes': 'root_diameter',
    'Textura': 'texture',
    'Tipo de estrutura': 'structure_type',
    'Classes de estrutura': 'structure_class',
    'Grau de estrutura': 'structure_degree',
    'Diâmetro de poros': 'pore_diameter',
    'Quantidade de poros': 'pore_quantity',
    'Forma de poros': 'pore_shape',
    'Cor (s)': 'dry_color_name',
    'Matiz (s)': 'dry_hue',
    'Valor (s)':'dry_value',
    'Croma (s)': 'dry_chroma',
    'Cor (h)': 'moist_color_name',
    'Matiz (h)': 'moist_hue',
    'Valor (h)': 'moist_value',
    'Croma (h)': 'moist_chroma',
    'Compacidade':'compaction',
    'Dureza': 'durability'
}, inplace=True)

# Drop unnecessary columns
morphology_cleaning = morphology.drop(columns=[
    'ID1', 'Agrupamento', 'REF', 'Pro', 'Observaçoes', 'Horizonte de diagnóstico', 'Propriedade de diagnóstico', 'Nitidez do limite', 'Designação do horizonte', 'Observaçoes', 'Confirmar', 'Adesividade', 'Plasticidade', 'Efervescência com HCl', 'Friabilidade', 'Orientação das Fendas', 'Largura das fendas', 'Quantidade de fendas'
])

#add profile_record_id to populate later from soil profile table
morphology_cleaning['profile_record_id'] = pd.NA

#drop accents
import unicodedata
def remove_accents(text):
    if isinstance(text, str):
        # Normalize and remove diacritics
        text = unicodedata.normalize('NFKD', text)
        text = ''.join(c for c in text if not unicodedata.combining(c))
        return text
    return text

# Apply to all cells in the DataFrame
morphology_cleaning = morphology_cleaning.applymap(remove_accents)

#morphology_cleaning = morphology_cleaning.fillna('null')

morphology_cleaning['profile'] = morphology_cleaning['profile'].astype(str).str.strip().str[:20]


# Reorder columns to match SAMPLES schema
morphology_clean = morphology_cleaning[[
    'horizon_id',
    'sample_id',
    'profile_record_id',
    'profile',
    'horizon_layer',
    'upper_depth',
    'lower_depth',
    'moisture_degree',
    'root_quantity',
    'root_diameter',
    'texture',
    'structure_type',
    'structure_class',
    'structure_degree',
    'pore_diameter',
    'pore_quantity',
    'pore_shape',
    'dry_color_name',
    'dry_hue',
    'dry_value',
    'dry_chroma',
    'moist_color_name',
    'moist_hue',
    'moist_value',
    'moist_chroma',
    'compaction',
    'durability'
]]

# Show first few rows
morphology_clean.head(5)


,horizon_id,sample_id,profile_record_id,profile,horizon_layer,upper_depth,lower_depth,moisture_degree,root_quantity,root_diameter,...,dry_color_name,dry_hue,dry_value,dry_chroma,moist_color_name,moist_hue,moist_value,moist_chroma,compaction,durability
0,B_101/62_1_1,10999.0,<NA>,101/62,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
1,B_101/62_2_1,11000.0,<NA>,101/62,2.0,11.0,28.0,Seco,Bastantes finas e medias e raras grossas,NaN,...,Pardo,10YR,5.0,3.0,Pardo-amarelado-escuro,10YR,3.0,4.0,Pequena,Brando
2,B_101/62_3_1,11001.0,<NA>,101/62,3.0,28.0,54.0,Seco,Algumas finas e medias e raras grossas,NaN,...,Pardo-amarelado-claro,10YR,6.0,4.0,Pardo-amarelado-escuro,10YR,4.0,4.0,Pequena a minima,Brando
3,B_101/62_4_1,11002.0,<NA>,101/62,4.0,54.0,90.0,Seco,"Poucas finas, algumas medias e raras grossas",NaN,...,Amarelo a amarelo-avermelhado,"8,75YR",7.0,6.0,Pardo-forte,"7,5YR",5.0,6.0,Pequena a minima,Brando
4,B_101/62_5_2,11003.2,<NA>,101/62,5.0,90.0,160.0,Seco a humido,Raras,Medias e grossas,...,Pardo-avermelhado,"7,5YR",7.0,6.0,Pardo-forte,"7,5YR",5.0,6.0,Pequena,Brando


# Soil Type Table

In [16]:
# Rename columns
soil_type = soil_profile.copy()

soil_type.rename(columns={
    'Perfil': 'profile',
    'Agrupamento': 'grouping',
    'Pro': 'province',
    'País': 'country',
    'Local': 'location',
    'DATA': 'date',
    'CEP_NOME': 'CEP_NAME',
}, inplace=True)

# Drop unnecessary columns
soil_type_cleaning = soil_type.drop(columns=[
    'REF', 'province', 'country', 'location', 'DESCRITOR1', 'DESCRITOR2', 'DESCRITOR3',
    'date', 'Fase', 'D_INSERÇAO', 'Publicação', 'WRB_old', 'Missão'
], errors='ignore')  # use errors='ignore' in case some columns were already missing

# Add a new Primary Key ID column starting from 1
soil_type_cleaning.insert(0, 'soil_type_id', range(1, len(soil_type_cleaning) + 1))

# Drop accents
import unicodedata

def remove_accents(text):
    if isinstance(text, str):
        text = unicodedata.normalize('NFKD', text)
        return ''.join(c for c in text if not unicodedata.combining(c))
    return text

# Apply to all cells in the DataFrame
soil_type_cleaning = soil_type_cleaning.applymap(remove_accents)

soil_type_cleaning['profile'] = soil_type_cleaning['profile'].astype(str).str.strip().str[:20]


# Keep only relevant columns
soil_type_clean = soil_type_cleaning[[
    'soil_type_id',
    'profile',
    'CEP_GR',
    'CEP_NAME',
    'FAO'
]]

# Preview
soil_type_clean.head(2)


,soil_type_id,profile,CEP_GR,CEP_NAME,FAO
0,1,1/51,NaN,NaN,NaN
1,2,1/57,Aridicos,Aridicos com calcario Pardo-cinzentos,CLha


# Site info table

In [17]:
site_info_cleaning = profile_loc

site_info_cleaning.rename(columns={
    'PERFIL': 'profile',
    'X_COORD': 'X_coord',
    'Y_COORD': 'Y_coord', 
    'PRO': 'district'
}, inplace=True)

# Add a new Primary Key ID column starting from 1
site_info_cleaning.insert(0, 'site_info_id', range(1, len(site_info_cleaning) + 1))


# Add missing columns
site_info_cleaning['land_cover_id'] = pd.NA
site_info_cleaning['climate_id'] = pd.NA  
site_info_cleaning['geology_id'] = pd.NA
site_info_cleaning['topo_feature_id'] = pd.NA  
site_info_cleaning['sampling_date'] = pd.NA # might leave out...
site_info_cleaning['districts_id'] = pd.NA  

# Function to remove accents
def remove_accents(text):
    if isinstance(text, str):
        text = unicodedata.normalize('NFKD', text)
        return ''.join(c for c in text if not unicodedata.combining(c))
    return text

# Apply to all object (string) columns
for col in site_info_cleaning.select_dtypes(include='object').columns:
    site_info_cleaning[col] = site_info_cleaning[col].apply(remove_accents)

site_info_cleaning['profile'] = site_info_cleaning['profile'].astype(str).str.strip().str[:20]

site_info_clean = site_info_cleaning[[
    'site_info_id',
    'ID',
    'profile', # keep for merge?
    'X_coord',
    'Y_coord',
    'land_cover_id',
    'climate_id',
    'geology_id',
    'topo_feature_id',
    'sampling_date',
    'districts_id',
    'district'
]]


site_info_clean.head()

,site_info_id,ID,profile,X_coord,Y_coord,land_cover_id,climate_id,geology_id,topo_feature_id,sampling_date,districts_id,district
0,1,2770,1/57,12.161278,-15.222598,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Namibe
1,2,48,1/59,12.575775,-4.866986,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Cabinda
2,3,1618,1/61,15.098840,-11.225411,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Cuanza Sul
3,4,881,1/63,17.081955,-9.274587,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Malanje
4,5,1750,1/64,20.788116,-11.568683,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Moxico


Save clean table

# Climate table


In [18]:
# Load original Excel data
profile_loc = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Perfis_local.xlsx")

# Create a copy for cleaning
climate_features_cleaning = profile_loc.copy()

# Mapping from climate codes to descriptions
code_to_description = {
    "B1": "Húmido",
    "B2": "Húmido",
    "B3": "Húmido",
    "B4": "Húmido",
    "C1": "Sub-húmido seco",
    "C2": "Sub-húmido chuvoso",
    "D": "Semi-árido",
    "E": "Árido",
    "Aw": "Tropical chuvoso com estação seca no Inverno, de savana",
    "BSw": "Seco de estepe, com chuva predominante no Verão",
    "BWw": "Seco de deserto, com chuva predominante no Verão",
    "Cw": "Mesotérmico húmido com estação seca no Inverno",
    "ARe": "Arídico extremo",
    "ARf": "Arídico fraco",
    "ARt": "Arídico típico",
    "tUDs": "Tempúdico seco",
    "tUSh": "Tempústico húmido",
    "tUSt": "Tempústico típico",
    "TUDs": "Tropúdico seco",
    "TUSa": "Tropústico arídico",
    "TUSu": "Tropústico údico",
    "TUSt": "Tropustico típico",
    "H": "Hipertérmico",
    "iH": "Iso-Hipertérmico",
    "iT": "Iso-Térmico",
    "T": "Térmico"
}

# Replace climate codes with descriptions
for col in ["CL_THORNTH", "CL_KOPPEN", "REG_HÍDRIC", "REG_TÉRMIC"]:
    climate_features_cleaning[col] = climate_features_cleaning[col].replace(code_to_description)

# Function to average values like "21-22" -> 21.5
def average_range(value):
    if isinstance(value, str) and '-' in value:
        try:
            nums = [float(x.strip()) for x in value.split('-')]
            return sum(nums) / len(nums)
        except:
            return None
    try:
        return float(value)
    except:
        return None

# Apply to temperature and precipitation columns
climate_features_cleaning["TMA"] = climate_features_cleaning["TMA"].apply(average_range)
climate_features_cleaning["PMA"] = climate_features_cleaning["PMA"].apply(average_range)

# Rename columns for clarity
climate_features_cleaning.rename(columns={
    'PERFIL': 'profile',
    'CL_THORNTH': 'thornthwaite_climate',
    'CL_KOPPEN': 'koppen_climate',
    'TMA': 'mean_annual_temp',
    'PMA': 'mean_annual_precip',
    'REG_HÍDRIC': 'hydric_regime',
    'REG_TÉRMIC': 'thermal_regime'
}, inplace=True)

# Drop accents from text values
def remove_accents(text):
    if isinstance(text, str):
        text = unicodedata.normalize('NFKD', text)
        return ''.join(c for c in text if not unicodedata.combining(c))
    return text

climate_features_cleaning = climate_features_cleaning.applymap(remove_accents)

# Add primary key column
climate_features_cleaning.insert(0, 'climate_id', range(1, len(climate_features_cleaning) + 1))

# Replace empty strings or nulls in numeric columns with \N (Postgres null)
for col in ['mean_annual_temp', 'mean_annual_precip']:
    climate_features_cleaning[col] = climate_features_cleaning[col].replace(
        ['', ' ', 'NULL', None, pd.NA, pd.NaT, 'nan', float('nan')], ''
    )

# Select and reorder relevant columns for export
climate_features_clean = climate_features_cleaning[[
    'climate_id',
    'ID',
    'mean_annual_temp',
    'mean_annual_precip',
    'koppen_climate',
    'thornthwaite_climate',
    'hydric_regime',
    'thermal_regime'
]]

# Preview
climate_features_clean.head()


,climate_id,ID,mean_annual_temp,mean_annual_precip,koppen_climate,thornthwaite_climate,hydric_regime,thermal_regime
0,1,2770,,,NaN,Arido,NaN,NaN
1,2,48,,,NaN,Humido,NaN,NaN
2,3,1618,,,NaN,NaN,NaN,NaN
3,4,881,21.5,1500.0,"Tropical chuvoso com estacao seca no Inverno, ...",Humido,Tropustico udico,Iso-Hipertermico
4,5,1750,,,NaN,NaN,NaN,NaN


Check FK relationships and datatypes

In [19]:
# Check and print the datatypes of each column
print("Data types of each column in profile_clean:\n")
print(climate_features_clean.dtypes)

Data types of each column in profile_clean:

climate_id               int64
ID                       int64
mean_annual_temp        object
mean_annual_precip      object
koppen_climate          object
thornthwaite_climate    object
hydric_regime           object
thermal_regime          object
dtype: object


# Topo features table

In [20]:
# Create a copy for cleaning
topo_features_cleaning = profile_loc.copy()

# Rename columns for clarity
topo_features_cleaning.rename(columns={
    'PERFIL': 'profile',
    'TOPOGRAFIA': 'slope_code',
    'ALTITUDE': 'altitude',
}, inplace=True)

# Add missing columns
topo_features_cleaning['aspect'] = pd.NA  
topo_features_cleaning['land_surface_temp'] = pd.NA
topo_features_cleaning['dem_elevation'] = pd.NA  

# Add primary key column
topo_features_cleaning.insert(0, 'topo_features_id', range(1, len(topo_features_cleaning) + 1))

# Create slope class mapping dictionary
slope_code_to_description = {
    "D1": "Plano (Declives < 2%)",
    "D2": "Ondulado muito suave (Declives > 2% e < 3%)",
    "D3": "Ondulado suave (Declives > 3% e < 5%)",
    "D4": "Ondulado (Declives > 5% e < 8%)",
    "D5": "Acidentado (Declives > 8% e < 15%)",
    "D6": "Escarpado (Declives >15% e < 30%)",
    "D7": "Montanhoso (Declives > 30%)"
}

# Create a mapping DataFrame for slope classes
slope_classes_df = pd.DataFrame([
    {"slope_code": code, "slope_description": desc}
    for code, desc in slope_code_to_description.items()
])

#drop accents
import unicodedata
def remove_accents(text):
    if isinstance(text, str):
        # Normalize and remove diacritics
        text = unicodedata.normalize('NFKD', text)
        text = ''.join(c for c in text if not unicodedata.combining(c))
        return text
    return text

# Apply to all cells in the DataFrame
topo_features_cleaning = topo_features_cleaning.applymap(remove_accents)

# Final cleaned topo features table (referencing slope_code, not description)
topo_features_clean = topo_features_cleaning[[
    'topo_features_id',
    'ID',
   # 'profile', 
    'slope_code',
    'altitude',
    'aspect',
    'land_surface_temp',
    'dem_elevation'
]]

# Preview
topo_features_clean.head()


,topo_features_id,ID,slope_code,altitude,aspect,land_surface_temp,dem_elevation
0,1,2770,NaN,32.0,<NA>,<NA>,<NA>
1,2,48,D5,NaN,<NA>,<NA>,<NA>
2,3,1618,NaN,NaN,<NA>,<NA>,<NA>
3,4,881,D1,1210.0,<NA>,<NA>,<NA>
4,5,1750,NaN,NaN,<NA>,<NA>,<NA>


# Geological features table

In [21]:
# Load cleaned geo features data
geo_features_cleaning = pd.read_excel("/Users/inesschwartz/GreenDataScience/Thesis/tables_soil_database/Perfis_local.xlsx")

# Rename columns
geo_features_cleaning.rename(columns={
    'PERFIL': 'profile',
    'GEOLOGIA': 'geology_id',
    'LITOLOGIA': 'lithology_id',
    'LITOLOGIA_1954': 'lithology_1954_id',
}, inplace=True)

# Add primary key column
geo_features_cleaning.insert(0, 'geo_features_id', range(1, len(geo_features_cleaning) + 1))

# Final normalized geo_features table (with codes as foreign keys)
geo_features_clean = geo_features_cleaning[[
    'geo_features_id',
   # 'profile', don't need fk
    'ID',
    'geology_id',
    'lithology_id',
    'lithology_1954_id'
]]

#drop accents
import unicodedata
def remove_accents(text):
    if isinstance(text, str):
        # Normalize and remove diacritics
        text = unicodedata.normalize('NFKD', text)
        text = ''.join(c for c in text if not unicodedata.combining(c))
        return text
    return text

# Apply to all cells in the DataFrame
geo_features_clean = geo_features_clean.applymap(remove_accents)

#preview
geo_features_clean.head()

,geo_features_id,ID,geology_id,lithology_id,lithology_1954_id
0,1,2770,NaN,d,NaN
1,2,48,Oendolongo,pp,Sistema do Maiombe
2,3,1618,NaN,NaN,NaN
3,4,881,Karroo,Cs/Cal,Serie de Cassanje - T2'T1
4,5,1750,NaN,NaN,NaN


In [22]:
## mapping tables
# Mappings for geology
geology_mapping = {
    "Kalahari": "Sistema do Kalahari",
    "Superficiais": "Formações Superficiais",
    "Karroo": "Sistema do Karroo",
    "Bembe": "Sistema do Bembe",
    "Oendolongo": "Sistema do Oendolongo",
    "Base": "Complexo de base",
    "Proterozóico": "Proterozóico",
    "Pleistocénico": "Pleistocénico",
    "Terciário": "Terciário (médio e inferior)",
    "TQ": "Quaternário e Terciário superior",
    "Cretácio": "",  # no description provided
    "RPKS": "Recente Plistocénico e Kalahari Superior"
}

# Mappings for lithology_1954
lithology_1954_mapping = {
    "γ": "Granitos, Granodioritos e Quartzodioritos",
    "PL": "Xistos, metaquartzitos, conglomerados, arcoses, ect.",
    "λ": "Rochas eruptivas indeterminadas",
    "δp": "Doleritos, doleritos pigeoníticos",
    "δab": "Diabases, diabases albito-cloriticas",
    "ε": "Noritos, gabros e peridotitos",
    "JK": "Composto de conglomerados, areias, cascalhos do Kalahari",
    "C": "Série Xisto - calcária",
    "Cal": "Sedimentos arenosos não consolidados",
    "K": "Série xisto - gresosa",
    "RT": "Não diferenciado",
    "σ": "Sienitos, sienitos nefelínicos",
    "Q": "Depósitos fossilíferos",
    "CS": "Grande conglomerado e série de Mwashya"
}

# Mappings for lithology
lithology_mapping = {
    "a": "Rochas arenáceas consolidadas",
    "aq": "Grés quartzíticos do Oendolongo",
    "b": "Rochas eruptivas básicas",
    "c": "Rochas sedimentares consolidadas calcárias",
    "c'": "Rochas sedimentares não consolidadas calcárias",
    "cg": "Rochas cristalofílicas argiláceas",
    "d": "Sedimentos não consolidados de origem marinha",
    "dc": "Depósitos coluvionares",
    "dr": "Diorítos",
    "e": "Rochas sedimentares consolidadas não calcárias",
    "g": "Rochas argiláceas consolidadas não calcárias",
    "g'": "Rochas argiláceas não consolidadas não calcárias",
    "g''": "Rochas cristalinas pouco micas em quartzo",
    "gp": "Rochas do  complexo gabro-plagioclastíco",
    "k": "Sedimentos não consolidados grosseiros do Kalahari",
    "m": "Rochas sedimentares não consolidadas calco-gipsíferas",
    "m'": "Depósitos coluvionares margosos",
    "mm": "Materiais mistos",
    "n": "Sedimentos não consolidados de origem continental",
    "nd": "não descrito",
    "pp": "Sedimentos não consolidados grosseiros plio-plistocénicos",
    "q": "Rochas cristalinas quartzíferas",
    "q'": "Materiais redistribuídos provenientes de desagregação rochas crist. quartzíferas",
    "qf": "Quartzitos ferruginosos do Oendolongo",
    "r": "Sedimentos grosseiros não especificados",
    "s": "Sienitos",
    "sx": "Formações (ou rochas) sedimentares não especificadas",
    "sx1": "Rochas sedimentares consolidadas com e sem calcário",
    "sx2": "Rochas sedimentares consolidadas",
    "v": "Materiais vulcânicos",
    "v'": "Rochas do complexo alcalino e/ou carboatítico",
    "x": "Rochas consolidadas não especificadas",
    "xm": "Xistos metamórficos",
    "xq": "Rochas cristalinas não especificadas",
    "z": "Rochas metassedimentares"
}

# Save each mapping as a DataFrame
pd.DataFrame([
    {"geology_code": k, "geology_description": v}
    for k, v in geology_mapping.items()
]).to_csv("/Users/inesschwartz/GreenDataScience/Thesis/tables_clean/geology_mapping.csv", index=False)

pd.DataFrame([
    {"lithology_code": k, "lithology_description": v}
    for k, v in lithology_mapping.items()
]).to_csv("/Users/inesschwartz/GreenDataScience/Thesis/tables_clean/lithology_mapping.csv", index=False)

pd.DataFrame([
    {"lithology_1954_code": k, "lithology_1954_description": v}
    for k, v in lithology_1954_mapping.items()
]).to_csv("/Users/inesschwartz/GreenDataScience/Thesis/tables_clean/lithology1954_mapping.csv", index=False)


# District table

**don't need**

Why do I need to make a separate table??

In [23]:
profile_loc.head()

,ID,PERFIL,X_COORD,Y_COORD,LATITUDE,LTGRAUS,LTMIN,LTSEC,LONGITUDE,LGGRAUS,...,ALTITUDE,GEOLOGIA,LITOLOGIA,LITOLOGIA_1954,CL_THORNTH,CL_KOPPEN,PMA,TMA,REG_HÍDRIC,REG_TÉRMIC
0,2770,1/57,12.161278,-15.222598,"-15,20108696",-15.0,12.0,3.9,"12,15307971",12.0,...,32.0,NaN,d,NaN,E,NaN,NaN,NaN,NaN,NaN
1,48,1/59,12.575775,-4.866986,NaN,0.0,0.0,0.0,NaN,0.0,...,NaN,Oendolongo,pp,Sistema do Maiombe,B1,NaN,NaN,NaN,NaN,NaN
2,1618,1/61,15.098840,-11.225411,NaN,0.0,0.0,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,881,1/63,17.081955,-9.274587,"-9,27458667755",0.0,0.0,0.0,"17,08195495605",0.0,...,1210.0,Karroo,Cs/Cal,Série de Cassanje - T2'T1,B3,Aw,1400-1600,21-22,TUSu,iH
4,1750,1/64,20.788116,-11.568683,NaN,0.0,0.0,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
import pandas as pd
import unicodedata

# Copy original DataFrame
district_cleaning = profile_loc

# Rename 'Pro' to 'district'
district_cleaning.rename(columns={'PRO': 'district'}, inplace=True)

# Add primary key column
district_cleaning.insert(0, 'district_id', range(1, len(district_cleaning) + 1))

# Select relevant columns
district_clean = district_cleaning[[
    'district_id',
    'district'
]].copy()

# Strip whitespace from 'district'
district_clean['district'] = district_clean['district'].astype(str).str.strip()

# Function to remove accents
def remove_accents(text):
    if isinstance(text, str):
        text = unicodedata.normalize('NFKD', text)
        return ''.join(c for c in text if not unicodedata.combining(c))
    return text

# Apply to all object (string) columns
for col in district_clean.select_dtypes(include='object').columns:
    district_clean[col] = district_clean[col].apply(remove_accents)

# Preview
district_clean.head()

,district_id,district
0,1,Namibe
1,2,Cabinda
2,3,Cuanza Sul
3,4,Malanje
4,5,Moxico


# Minerology info table

# Biology Info table

# Checking CSV Datatypes before DB export

In [25]:
import pandas as pd
import os

# Set the path to your folder containing the 10 CSV files
csv_folder = "/Users/inesschwartz/GreenDataScience/Thesis/tables_clean"  

# List all CSV files in the folder
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

# Loop through each file and display column data types
for file in csv_files:
    file_path = os.path.join(csv_folder, file)
    print(f"\n📄 File: {file}")
    
    try:
        df = pd.read_csv(file_path)
        print(df.dtypes)
    except Exception as e:
        print(f"⚠️ Error reading {file}: {e}")


📄 File: soil_type_clean.csv
soil_type_id     int64
profile         object
CEP_GR          object
CEP_NAME        object
FAO             object
dtype: object

📄 File: analyses_clean.csv
lab_sample_id      int64
sample_id        float64
EG               float64
thick_clay       float64
fine_clay        float64
                  ...   
Tl               float64
Pb               float64
Bi               float64
Th               float64
U                float64
Length: 68, dtype: object

📄 File: geology_mapping.csv
geology_code           object
geology_description    object
dtype: object

📄 File: topo_feat_clean.csv
topo_features_id       int64
slope_code            object
altitude             float64
aspect               float64
land_surface_temp    float64
dem_elevation        float64
dtype: object

📄 File: lithology1954_mapping.csv
lithology_1954_code           object
lithology_1954_description    object
dtype: object

📄 File: profile_record_clean.csv
profile_record_id      int64
profile

/var/folders/tp/79mdnyy56_xc3g1jvp9wf4_80000gn/T/ipykernel_3633/3654665737.py:16: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


# Foreign key imports and datatype consistency

In [26]:
## standardize datatypes for identifiers
def convert_identifiers_to_string(df, id_columns):
    """
    Converts specified identifier columns in a DataFrame to string type,
    safely handling float64 values and preserving missing values (NA).
    Avoids SettingWithCopyWarning.
    """
    df = df.copy()  # ensure we're working with a copy, not a slice
    for col in id_columns:
        if col in df.columns:
            df.loc[:, col] = df[col].apply(
                lambda x: str(int(x)) if pd.notna(x) and isinstance(x, float) and x.is_integer()
                else str(x) if pd.notna(x)
                else pd.NA
            ).astype("string")
    return df

#usage

# Define identifier columns
identifier_columns = [
    'sample_id', 'site_info_id', 'profile',
    'horizon_id', 'lab_sample_id'
]

# Apply to each relevant dataframe
samples_check = convert_identifiers_to_string(samples_check, identifier_columns)
morphology_clean = convert_identifiers_to_string(morphology_clean, identifier_columns)
profile_clean = convert_identifiers_to_string(profile_record_clean, identifier_columns)
merged1 = convert_identifiers_to_string(merged1, identifier_columns)
site_info_clean = convert_identifiers_to_string(site_info_clean, identifier_columns)
#district_clean = convert_identifiers_to_string(district_clean, identifier_columns)
soil_type_clean = convert_identifiers_to_string(soil_type_clean, identifier_columns)
geo_features_clean = convert_identifiers_to_string(geo_features_clean, identifier_columns)
topo_features_clean = convert_identifiers_to_string(topo_features_clean, identifier_columns)


In [27]:
# Check types of identifier columns in each dataframe
dfs = {
    "samples_check": samples_check,
    "morphology_clean": morphology_clean,
    "profile_clean": profile_clean,
    "merged1": merged1,
    "site_info_clean": site_info_clean,
    #"district_clean": district_clean,
    "soil_type_clean": soil_type_clean,
    "geo_features_clean": geo_features_clean,
    "topo_features_clean": topo_features_clean,
}

for name, df in dfs.items():
    print(f"\n{name} column types:")
    for col in identifier_columns:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")



samples_check column types:
  sample_id: object
  site_info_id: object
  profile: object
  horizon_id: object

morphology_clean column types:
  sample_id: string
  profile: object
  horizon_id: object

profile_clean column types:
  sample_id: object
  site_info_id: object
  profile: object

merged1 column types:
  sample_id: object
  lab_sample_id: object

site_info_clean column types:
  site_info_id: string
  profile: object

soil_type_clean column types:
  profile: object

geo_features_clean column types:

topo_features_clean column types:


In [28]:
## validate FK references in tables

# Check for missing FK references in morphology
missing_samples = samples_check[~samples_check['sample_id'].isin(morphology_clean['sample_id'])]

print(f"{len(missing_samples)} sample_id(s) in samples_clean are missing in morphology_horizon_clean.")
print(missing_samples[['sample_id']])


9636 sample_id(s) in samples_clean are missing in morphology_horizon_clean.
      sample_id
3           633
10          700
11          707
13          710
16          719
...         ...
14710     18867
14711     18868
14712     18869
14713     18870
14714     18871

[9636 rows x 1 columns]


In [29]:
# Samples table horizon_id FK 

# Drop old horizon_id if it exists (to avoid confusion)
samples_check = samples_check.drop(columns=['horizon_id'], errors='ignore')

# Merge horizon_id from morph into samples
samples_check = samples_check.merge(
    morphology_clean[['sample_id', 'horizon_id']],
    on='sample_id',
    how='left'
)

missing = samples_check[samples_check['horizon_id'].isna()]
print(f"{len(missing)} samples could not be matched with a horizon_id.")


9636 samples could not be matched with a horizon_id.


In [30]:

# Re-order and select relevant columns
samples_clean = samples_check[[
    'sample_id',
    'site_info_id', 
    'profile',
    'horizon_id',
    'year',
    'shelf',
    'room'  # Ensure this matches the column name in your DataFrame
]].copy()

# Preview the result
samples_clean.head()


,sample_id,site_info_id,profile,horizon_id,year,shelf,room
0,630,172,139,Hb_139/46_1_1,1946.0,1,22
1,631,173,139,Hb_139/46_2_1,1946.0,1,22
2,632,174,139,Hb_139/46_3_1,1946.0,1,22
3,633,175,139,NaN,1946.0,1,22
4,687,1034,208,Hb_208/46_1_1,1946.0,1,22


In [31]:
samples_check.head()

,sample_id,site_info_id,profile_record_id,profile,shelf,room,year,horizon_id
0,630,172,<NA>,139,1,22,1946.0,Hb_139/46_1_1
1,631,173,<NA>,139,1,22,1946.0,Hb_139/46_2_1
2,632,174,<NA>,139,1,22,1946.0,Hb_139/46_3_1
3,633,175,<NA>,139,1,22,1946.0,NaN
4,687,1034,<NA>,208,1,22,1946.0,Hb_208/46_1_1


## completing profile_record_clean table

In [32]:
profile_record_clean.head()

,profile_record_id,profile,site_info_id,sample_id,soil_type_id
0,1,1/51,<NA>,<NA>,<NA>
1,2,1/57,<NA>,<NA>,<NA>
2,3,1/59,<NA>,<NA>,<NA>
3,4,1/63,<NA>,<NA>,<NA>
4,5,10/54,<NA>,<NA>,<NA>


In [33]:
# Drop old foreign key columns if they exist
profile_record_clean = profile_record_clean.drop(
    columns=['site_info_id', 'sample_id', 'soil_type_id'], errors='ignore'
)

# Merge site_info_id (one-to-one or many-to-one) using 'profile'
profile_record_clean = profile_record_clean.merge(
    site_info_clean[['profile', 'site_info_id']],
    on='profile',
    how='left'
)

# Merge soil_type_id (one-to-one or many-to-one) using 'profile'
profile_record_clean = profile_record_clean.merge(
    soil_type_clean[['profile', 'soil_type_id']],
    on='profile',
    how='left'
)

# Merge sample_id (one-to-one or many-to-one) using 'profile'
profile_record_clean = profile_record_clean.merge(
    samples_clean[['profile', 'sample_id']],
    on='profile',
    how='left'
)

#drop PK and replace w new profile_record_id PK
# Drop old FK columns if they exist
profile_record_clean = profile_record_clean.drop(
    columns=['profile_record_id' ],
    errors='ignore'
)

# Ensure primary key 'profile_record_id' exists
if 'profile_record_id' not in profile_record_clean.columns:
    profile_record_clean.insert(0, 'profile_record_id', range(1, len(profile_record_clean) + 1))

profile_record_clean['profile_record_id'] = profile_record_clean['profile_record_id'].astype(str)

# Final column order
column_order = [
    'profile_record_id',
    'profile',
    'site_info_id',
    'soil_type_id',
    'sample_id'
]
profile_record_clean = profile_record_clean[column_order]

# Check for missing links (optional sanity check)
missing_site = profile_record_clean[profile_record_clean['site_info_id'].isna()]
missing_soil = profile_record_clean[profile_record_clean['soil_type_id'].isna()]

print(f"{len(missing_site)} profiles missing site_info_id")
print(f"{len(missing_soil)} profiles missing soil_type_id")

profile_record_clean.head()

155 profiles missing site_info_id
0 profiles missing soil_type_id


,profile_record_id,profile,site_info_id,soil_type_id,sample_id
0,1,1/51,<NA>,1,NaN
1,2,1/57,1,2,4835
2,3,1/57,1,2,4836
3,4,1/57,1,2,4837
4,5,1/57,1,2,4838


In [34]:
morphology_clean.head()

,horizon_id,sample_id,profile_record_id,profile,horizon_layer,upper_depth,lower_depth,moisture_degree,root_quantity,root_diameter,...,dry_color_name,dry_hue,dry_value,dry_chroma,moist_color_name,moist_hue,moist_value,moist_chroma,compaction,durability
0,B_101/62_1_1,10999,<NA>,101/62,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
1,B_101/62_2_1,11000,<NA>,101/62,2.0,11.0,28.0,Seco,Bastantes finas e medias e raras grossas,NaN,...,Pardo,10YR,5.0,3.0,Pardo-amarelado-escuro,10YR,3.0,4.0,Pequena,Brando
2,B_101/62_3_1,11001,<NA>,101/62,3.0,28.0,54.0,Seco,Algumas finas e medias e raras grossas,NaN,...,Pardo-amarelado-claro,10YR,6.0,4.0,Pardo-amarelado-escuro,10YR,4.0,4.0,Pequena a minima,Brando
3,B_101/62_4_1,11002,<NA>,101/62,4.0,54.0,90.0,Seco,"Poucas finas, algumas medias e raras grossas",NaN,...,Amarelo a amarelo-avermelhado,"8,75YR",7.0,6.0,Pardo-forte,"7,5YR",5.0,6.0,Pequena a minima,Brando
4,B_101/62_5_2,11003.2,<NA>,101/62,5.0,90.0,160.0,Seco a humido,Raras,Medias e grossas,...,Pardo-avermelhado,"7,5YR",7.0,6.0,Pardo-forte,"7,5YR",5.0,6.0,Pequena,Brando


In [35]:
# Convert to string dtype in both DataFrames before merge
morphology_clean['profile_record_id'] = morphology_clean['profile_record_id'].astype(str)
profile_record_clean['profile_record_id'] = profile_record_clean['profile_record_id'].astype(str)

#drop old FK
# Drop old FK columns if they exist
morphology_clean = morphology_clean.drop(
    columns=['profile_record_id'],
    errors='ignore'
)
# Now merge on 'profile_record_id'
morphology_clean = morphology_clean.merge(
    profile_record_clean[['profile', 'profile_record_id']],
    on='profile',
    how='left'
)

morphology_clean1 = morphology_clean[[
    'horizon_id',
    'sample_id',
    'profile_record_id',
    #'profile',
    'horizon_layer',
    'upper_depth',
    'lower_depth',
    'moisture_degree',
    'root_quantity',
    'root_diameter',
    'texture',
    'structure_type',
    'structure_class',
    'structure_degree',
    'pore_diameter',
    'pore_quantity',
    'pore_shape',
    'dry_color_name',
    'dry_hue',
    'dry_value',
    'dry_chroma',
    'moist_color_name',
    'moist_hue',
    'moist_value',
    'moist_chroma',
    'compaction',
    'durability'
]]

In [36]:
morphology_clean1.head()



,horizon_id,sample_id,profile_record_id,horizon_layer,upper_depth,lower_depth,moisture_degree,root_quantity,root_diameter,texture,...,dry_color_name,dry_hue,dry_value,dry_chroma,moist_color_name,moist_hue,moist_value,moist_chroma,compaction,durability
0,B_101/62_1_1,10999,56,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,Arenoso,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
1,B_101/62_1_1,10999,57,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,Arenoso,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
2,B_101/62_1_1,10999,58,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,Arenoso,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
3,B_101/62_1_1,10999,59,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,Arenoso,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando
4,B_101/62_1_1,10999,60,1.0,0.0,11.0,Seco,Muitas finas e bastantes medias,NaN,Arenoso,...,Pardo-acinzentado a pardo,10YR,5.0,2.5,Pardo-acinzentado-escuro,10YR,4.0,2.0,Pequena a minima,Brando


## Completing site_info_clean table

In [37]:
# Convert keys to string before merge to avoid dtype mismatch errors
site_info_clean['site_info_id'] = site_info_clean['ID'].astype(str)
geo_features_clean['ID'] = geo_features_clean['ID'].astype(str)
climate_features_clean['ID'] = climate_features_clean['ID'].astype(str)
topo_features_clean['ID'] = topo_features_clean['ID'].astype(str)

/var/folders/tp/79mdnyy56_xc3g1jvp9wf4_80000gn/T/ipykernel_3633/4452788.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  climate_features_clean['ID'] = climate_features_clean['ID'].astype(str)


In [38]:
# Drop old FK columns if they exist
site_info_clean = site_info_clean.drop(
    columns=['geo_features_id', 'climate_id', 'topo_features_id', 'district_id', 'land_cover_id', 'geology_id','topo_feature_id','sampling_date', 'districts_id' ],
    errors='ignore'
)

In [39]:
# --- Step 1: Convert 'ID' columns to string ---
site_info_clean['ID'] = site_info_clean['ID'].astype(str)
geo_features_clean['ID'] = geo_features_clean['ID'].astype(str)
climate_features_clean['ID'] = climate_features_clean['ID'].astype(str)
topo_features_clean['ID'] = topo_features_clean['ID'].astype(str)

# --- Step 2: Merge feature tables into site_info_clean ---
site_info_clean = site_info_clean.merge(
    geo_features_clean[['ID', 'geo_features_id']],
    on='ID',
    how='left'
)

site_info_clean = site_info_clean.merge(
    climate_features_clean[['ID', 'climate_id']],
    on='ID',
    how='left'
)

site_info_clean = site_info_clean.merge(
    topo_features_clean[['ID', 'topo_features_id']],
    on='ID',
    how='left'
)

# --- Step 3: Drop 'ID' column from all involved tables ---
for table in ['site_info_clean', 'geo_features_clean', 'climate_features_clean', 'topo_features_clean']:
    df = globals().get(table)
    if df is not None and 'ID' in df.columns:
        df = df.drop(columns=['ID'])
        globals()[table] = df
        print(f"✅ Dropped 'ID' from {table}")
    else:
        print(f"⏭️ Skipping {table}: not found or no 'ID' column")

# --- District: Skip merging district_id, use district name directly ---

# don't see need for district_id, just use district name
# For district (assuming district columns are already strings)
# site_info_clean = site_info_clean.merge(
#     district_clean[['district', 'district_id']],
#     on='district',
#     how='left'
# )


✅ Dropped 'ID' from site_info_clean
✅ Dropped 'ID' from geo_features_clean
✅ Dropped 'ID' from climate_features_clean
✅ Dropped 'ID' from topo_features_clean


/var/folders/tp/79mdnyy56_xc3g1jvp9wf4_80000gn/T/ipykernel_3633/2925909182.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  climate_features_clean['ID'] = climate_features_clean['ID'].astype(str)


In [40]:
## need to drop ID from climate, geo_features, topo_features, and site_info_clean

In [41]:
site_info_clean.head()

,site_info_id,profile,X_coord,Y_coord,district,geo_features_id,climate_id,topo_features_id
0,2770,1/57,12.161278,-15.222598,Namibe,1,1,1
1,48,1/59,12.575775,-4.866986,Cabinda,2,2,2
2,1618,1/61,15.098840,-11.225411,Cuanza Sul,3,3,3
3,881,1/63,17.081955,-9.274587,Malanje,4,4,4
4,1750,1/64,20.788116,-11.568683,Moxico,5,5,5


In [42]:
profile_record_clean.head()

,profile_record_id,profile,site_info_id,soil_type_id,sample_id
0,1,1/51,<NA>,1,NaN
1,2,1/57,1,2,4835
2,3,1/57,1,2,4836
3,4,1/57,1,2,4837
4,5,1/57,1,2,4838


In [43]:
## changing column headers to be SQL friendly
# Rename problematic columns
merged1 = merged1.rename(columns={
    'atm_1/3': 'atm_1_3',
    'Ca++': 'Ca',
    'Mg++': 'Mg',
    'Na+': 'Na',
    'K+': 'K',
    'Min_<0,002': 'Min_lt_0002',
    'Min_0,05-0,02': 'Min_005_002',
    'Min_0,2-0,05': 'Min_02_005',
    'Min_2-0,2': 'Min_2_02',
    'As': 'arsenic',
    'P': 'phosphorus',
    'S': 'sulfur',
    'V': 'vanadium'
})


# SAVING CLEANED TABLES TO CSV FOR DB

In [44]:
import pandas as pd

# Prepare dictionary of clean tables
tables = {
    "samples": samples_clean,
    "analyses": merged1,
    "profile_record": profile_record_clean,
    "morphology_horizon": morphology_clean1,
    "soil_type": soil_type_clean,
    "site_info": site_info_clean,
    "climate_feat": climate_features_clean,
    "topo_feat": topo_features_clean,
    "geo_feat": geo_features_clean,
    "districts": district_clean
}

# # Save each table to CSV with 'NULL' in empty cells
for name, df in tables.items():
#     df.replace("", pd.NA, inplace=True)  # Replace empty strings with NA
    df.to_csv(f"/Users/inesschwartz/GreenDataScience/Thesis/tables_clean/{name}_clean.csv", index=False)

In [45]:
# Save random 100 rows of each table as mini versions for DB test
import os

# Create folder if it doesn't exist
mini_path = "/Users/inesschwartz/GreenDataScience/Thesis/tables_clean_mini"
os.makedirs(mini_path, exist_ok=True)

# Save random 100 rows of each table as mini versions
samples_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/samples.csv", index=False)
merged1.sample(n=100, random_state=42).to_csv(f"{mini_path}/analyses.csv", index=False)
profile_record_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/profile_record.csv", index=False)
morphology_clean1.sample(n=100, random_state=42).to_csv(f"{mini_path}/morphology_horizon.csv", index=False)
soil_type_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/soil_type.csv", index=False)
site_info_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/site_info.csv", index=False)
climate_features_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/climate_feat.csv", index=False)
topo_features_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/topo_feat.csv", index=False)
geo_features_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/geo_feat.csv", index=False)
#district_clean.sample(n=100, random_state=42).to_csv(f"{mini_path}/districts.csv", index=False)



In [46]:
tables = {
    "samples": samples_clean,
    "analyses": merged1,
    "profile_record": profile_record_clean,
    "morphology_horizon": morphology_clean1,
    "soil_type": soil_type_clean,
    "site_info": site_info_clean,
    "climate_feat": climate_features_clean,
    "topo_feat": topo_features_clean,
    "geo_feat": geo_features_clean
    #"districts": district_clean
}

for name, df in tables.items():
#     df.replace("", pd.NA, inplace=True)
     df.to_csv(f"/Users/inesschwartz/GreenDataScience/Thesis/tables_clean_mini/{name}.csv", index=False)


In [47]:
# List your DataFrame variables here
dataframes = {
    "samples": samples_clean,
    "analyses": merged1,
    "profile_record": profile_record_clean,
    "morphology_horizon": morphology_clean1,
    "soil_type": soil_type_clean,
    "site_info": site_info_clean,
    "climate_feat": climate_features_clean,
    "topo_feat": topo_features_clean,
    "geo_feat": geo_features_clean
    #"districts": district_clean
}

# Print column names
for name, df in dataframes.items():
    print(f"\n{name} ({len(df)} rows):")
    for col in df.columns:
        print(f"  - {col}")



samples (14766 rows):
  - sample_id
  - site_info_id
  - profile
  - horizon_id
  - year
  - shelf
  - room

analyses (7847 rows):
  - lab_sample_id
  - sample_id
  - EG
  - thick_clay
  - fine_clay
  - silt
  - clay
  - Eq_Hum
  - atm_1_3
  - atm_15
  - CACO3
  - gypsum
  - free_iron
  - organic_carbon
  - total_N
  - P205
  - organic_material
  - pH_H2O
  - pH_KCL
  - Ca
  - Mg
  - Na
  - K
  - exchangable_bases_sum
  - CEC
  - vanadium
  - conductivity
  - soluble_sodium
  - Min_lt_0002
  - Min_005_002
  - Min_02_005
  - Min_2_02
  - field_sample_code
  - Depth
  - Al
  - Si
  - phosphorus
  - sulfur
  - Cl
  - Ti
  - Cr
  - Mn
  - Fe
  - Co
  - Ni
  - Cu
  - Zn
  - arsenic
  - Se
  - Rb
  - Sr
  - Zr
  - Nb
  - Mo
  - Cd
  - Sn
  - Sb
  - Ba
  - Ta
  - W
  - Pt
  - Au
  - Hg
  - Tl
  - Pb
  - Bi
  - Th
  - U

profile_record (7516 rows):
  - profile_record_id
  - profile
  - site_info_id
  - soil_type_id
  - sample_id

morphology_horizon (41205 rows):
  - horizon_id
  - sample_id
 

In [48]:
missing = profile_record_clean[~profile_record_clean['soil_type_id'].isin(soil_type_clean['soil_type_id'])]
print(missing)

Empty DataFrame
Columns: [profile_record_id, profile, site_info_id, soil_type_id, sample_id]
Index: []


In [49]:
profile_record_clean.head()

,profile_record_id,profile,site_info_id,soil_type_id,sample_id
0,1,1/51,<NA>,1,NaN
1,2,1/57,1,2,4835
2,3,1/57,1,2,4836
3,4,1/57,1,2,4837
4,5,1/57,1,2,4838


In [50]:
soil_type_clean.head()

,soil_type_id,profile,CEP_GR,CEP_NAME,FAO
0,1,1/51,NaN,NaN,NaN
1,2,1/57,Aridicos,Aridicos com calcario Pardo-cinzentos,CLha
2,3,1/59,Psamoferralicos,"Psamo-ferralicos Amarelos ou Alaranjados, sedi...",FRxa
3,4,1/63,Ferraliticos,Fracamente Ferralicos Vermelhos Clino-argilico...,FRh
4,5,10/54,Ferraliticos,Fracamente Ferralicos pardo-amarelados,FRh


In [51]:
# Get list of valid soil_type_ids
valid_soil_type_ids = set(soil_type_clean['soil_type_id'])

# Find invalid soil_type_ids in profile_df
invalid_rows = profile_record_clean[~profile_record_clean['soil_type_id'].isin(valid_soil_type_ids)]

# Show the result
if not invalid_rows.empty:
    print("❌ These rows reference soil_type_id values that do NOT exist in soil_type.csv:")
    print(invalid_rows)
else:
    print("✅ All soil_type_id values in profile_record.csv are valid.")


✅ All soil_type_id values in profile_record.csv are valid.


In [52]:
soil_type_ids = set(soil_type_clean['soil_type_id'].unique())
profile_soil_type_ids = set(profile_record_clean['soil_type_id'].unique())

missing_ids = profile_soil_type_ids - soil_type_ids

print("soil_type_ids in profile_record missing from soil_type.csv:", missing_ids)


soil_type_ids in profile_record missing from soil_type.csv: set()


In [53]:
# Check if 4835 is in soil_type_id column of soil_type.csv
soil_type_has_4835 = 4835 in soil_type_clean['soil_type_id'].values

# Check if 4835 is in soil_type_id column of profile_record.csv
profile_record_has_4835 = 4835 in profile_record_clean['soil_type_id'].values

print(f"4835 in soil_type.csv? {soil_type_has_4835}")
print(f"4835 in profile_record.csv? {profile_record_has_4835}")

4835 in soil_type.csv? False
4835 in profile_record.csv? False


In [59]:
print(4835 in profile_record_clean['soil_type_id'].values)

False


In [60]:
print(4835 in profile_record_clean['soil_type_id'].values)
print(profile_record_clean[profile_record_clean['soil_type_id'] == 4835])

False
Empty DataFrame
Columns: [profile_record_id, profile, site_info_id, soil_type_id, sample_id]
Index: []


In [61]:
soil_type_ids_in_soil_type = set(soil_type_clean['soil_type_id'].unique())
soil_type_ids_in_profile_record = set(profile_record_clean['soil_type_id'].unique())

missing_soil_type_ids = soil_type_ids_in_profile_record - soil_type_ids_in_soil_type

print(f"Missing soil_type_ids in soil_type.csv: {missing_soil_type_ids}")


Missing soil_type_ids in soil_type.csv: set()
